# ARC-AGI-3 Solver — Qwen3.8-27B-FP8 — 25-Game P1 Public Eval

Public/offline evaluation is overridden to the same 25 public games × 1 pass shape. Competition reruns still use the live private game list from the Kaggle gateway.


In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

In [ ]:
# Qwen3.8 / Kaggle input configuration
DATASET_SOURCES: list[str] = [
    "jakobbrggen/taaf-kaggle-source-anim-20260807-anim",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
]
KERNEL_SOURCES: list[str] = []

# New private Kaggle Model (Version 1).
QWEN_MODEL_OWNER = "foysalemonshanto"
QWEN_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN_MODEL_REF = f"{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}"
QWEN_MODEL_VARIATION = "hf-fp8"
QWEN_MODEL_VERSION = "1"
QWEN_SERVED_MODEL_NAME = "Qwen/Qwen3.8-27B-FP8"
QWEN_MODEL_PATH = Path(
    f"/kaggle/input/models/{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}/"
    f"pytorch/{QWEN_MODEL_VARIATION}/{QWEN_MODEL_VERSION}"
)

DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path

    # Prefer the attached bundle whose marker actually exists.
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent

    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(
        json.dumps(data, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Verify the Qwen3.8 Kaggle Model before any expensive setup work starts.
if not QWEN_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        "Qwen3.8 Kaggle Model is not attached.\n"
        f"Expected path:\n{QWEN_MODEL_PATH}\n\n"
        "Attach: Qwen3.8 27B FP8 Repacked → PyTorch → hf-fp8 → Version 1"
    )

_required_qwen_files = [
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "outside.safetensors",
    "mtp.safetensors",
    "chat_template.jinja",
]
_missing_qwen_files = [
    name for name in _required_qwen_files if not (QWEN_MODEL_PATH / name).is_file()
]
if _missing_qwen_files:
    raise FileNotFoundError(
        "Qwen3.8 mount is incomplete; missing: " + ", ".join(_missing_qwen_files)
    )

_qwen_layer_shards = sorted(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))
_qwen_safetensors = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
if len(_qwen_layer_shards) != 16 or len(_qwen_safetensors) != 18:
    raise RuntimeError(
        "Unexpected Qwen3.8 checkpoint layout: "
        f"{len(_qwen_layer_shards)} layer shards, "
        f"{len(_qwen_safetensors)} safetensors files."
    )

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# The bundled setup resolver asks for owner/slug. Give it a model ref that maps
# directly to the full Kaggle Model version directory.
kaggle_input_paths[QWEN_MODEL_REF] = str(QWEN_MODEL_PATH)

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_QWEN_MODEL_REF": QWEN_MODEL_REF,
    "TAAF_QWEN_MODEL_PATH": str(QWEN_MODEL_PATH),
    "TAAF_QWEN_SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)

print("\n✅ Qwen3.8 input configuration ready")
print(f"Model ref:       {QWEN_MODEL_REF}")
print(f"Physical path:   {QWEN_MODEL_PATH}")
print(f"Served model:    {QWEN_SERVED_MODEL_NAME}")
print(f"Safetensors:     {len(_qwen_safetensors)}")
print(f"Layer shards:    {len(_qwen_layer_shards)}")
print(f"TAAF input map:  {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# Audit the attached inputs that matter for this run.
print("=== TAAF bundle ===")
print(BUNDLE_DIR)
print("Exists:", BUNDLE_DIR.exists())

print("\n=== vLLM wheelhouse ===")
_vllm_wheelhouse = Path(
    "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
)
print(_vllm_wheelhouse)
print("Exists:", _vllm_wheelhouse.exists())

print("\n=== Qwen3.8 Kaggle Model ===")
print(QWEN_MODEL_PATH)
print("Exists:", QWEN_MODEL_PATH.exists())
print("Safetensors:", len(list(QWEN_MODEL_PATH.glob("*.safetensors"))))
print(
    "Repacked layer shards:",
    len(list(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))),
)


In [ ]:
import re


def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []

    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"
    env.update(_load_setup_env())
    return env


def _replace_python_assignment(
    command: str,
    variable_name: str,
    value: str,
) -> tuple[str, int]:
    """Replace a top-level Python string assignment inside the setup here-doc."""
    pattern = rf"(?m)^{re.escape(variable_name)}\s*=\s*(['\"])[^\r\n]*?\1\s*$"
    replacement = f"{variable_name} = {value!r}"
    return re.subn(pattern, replacement, command, count=1)


def _patch_qwen38_setup_commands(commands: list[str]) -> list[str]:
    """
    Preserve the TAAF deployment setup but replace its model identity with the
    Qwen3.8 Kaggle Model. This avoids copying/forking the large bundled setup
    script and keeps the wheelhouse/GPU/vLLM behavior from the source bundle.
    """
    patched: list[str] = []
    replacement_counts = {
        "MODEL_OWNER": 0,
        "MODEL_SLUG": 0,
        "SERVED_MODEL_NAME": 0,
    }

    replacements = {
        "MODEL_OWNER": QWEN_MODEL_OWNER,
        "MODEL_SLUG": QWEN_MODEL_SLUG,
        "SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    }

    for raw_command in commands:
        command = str(raw_command)

        for variable_name, value in replacements.items():
            command, count = _replace_python_assignment(
                command,
                variable_name,
                value,
            )
            replacement_counts[variable_name] += count

        # Make offline behavior explicit in the child process as well.
        if "def vllm_env()" in command:
            command = command.replace(
                "'VLLM_NO_USAGE_STATS': '1',",
                "'VLLM_NO_USAGE_STATS': '1',\n"
                "            'HF_HUB_OFFLINE': '1',\n"
                "            'TRANSFORMERS_OFFLINE': '1',",
                1,
            )

        patched.append(command)

    missing = [
        name for name, count in replacement_counts.items() if count == 0
    ]
    if missing:
        raise RuntimeError(
            "Could not update the bundled TAAF setup for Qwen3.8. "
            "Missing assignment(s): "
            + ", ".join(missing)
            + ". The attached TAAF bundle's setup_commands.json has changed."
        )

    print("taaf.kaggle: Qwen3.8 setup patch =", replacement_counts, flush=True)
    return patched


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return

    commands = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(commands, list):
        raise RuntimeError(f"{path} must contain a JSON list of shell commands.")

    if filename == "setup_commands.json":
        commands = _patch_qwen38_setup_commands(commands)

    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(
            str(command),
            shell=True,
            check=check,
            cwd=WORKING_DIR,
            env=env,
        )
        if not check and result.returncode != 0:
            print(
                f"taaf.kaggle: {label} command exited with {result.returncode}",
                flush=True,
            )

        # Setup commands may export additional runtime settings.
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text(
        "".join(f"{entry}\n" for entry in source_entries),
        encoding="utf-8",
    )
    print(
        f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)",
        flush=True,
    )

# Run the TAAF deployment setup, patched to use Qwen3.8.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [
    entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry
]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Fail early if the analyzer is still exposing an old model identity.
_actual_model_id = os.environ.get("INFERENCE_ANALYZER_MODEL", "")
if _actual_model_id != QWEN_SERVED_MODEL_NAME:
    raise RuntimeError(
        "TAAF setup completed, but the analyzer model ID is wrong: "
        f"{_actual_model_id!r}; expected {QWEN_SERVED_MODEL_NAME!r}"
    )

print("\n✅ TAAF/vLLM setup completed for Qwen3.8")
print("Model path:", QWEN_MODEL_PATH)
print("Analyzer model:", _actual_model_id)
print("Analyzer endpoint:", os.environ.get("LOCAL_ANALYZER_BASE_URL"))


In [ ]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Inline customization hook — Q38 P1-style public evaluation.
#
# Q38 P1 runs the full 25 public ARC-AGI-3 games once each (25 games × 1 pass).
# This override applies only to the public/offline notebook run. Competition reruns
# still replace bm.games from Kaggle's live gateway in the final run cell.

print("Benchmark analyzer model:", os.environ.get("INFERENCE_ANALYZER_MODEL"))
print("Qwen3.8 model path:", os.environ.get("TAAF_QWEN_MODEL_PATH"))

Q38_P1_PUBLIC_GAME_IDS = [
    "ar25-0c556536",
    "bp35-0a0ad940",
    "cd82-fb555c5d",
    "cn04-2fe56bfb",
    "dc22-fdcac232",
    "ft09-0d8bbf25",
    "g50t-5849a774",
    "ka59-38d34dbb",
    "lf52-271a04aa",
    "lp85-305b61c3",
    "ls20-9607627b",
    "m0r0-492f87ba",
    "r11l-495a7899",
    "re86-8af5384d",
    "s5i5-18d95033",
    "sb26-7fbdac44",
    "sc25-635fd71a",
    "sk48-d8078629",
    "sp80-589a99af",
    "su15-1944f8ab",
    "tn36-ef4dde99",
    "tr87-cd924810",
    "tu93-0768757b",
    "vc33-5430563c",
    "wa30-ee6fef47",
]

if not true_submission:
    if len(Q38_P1_PUBLIC_GAME_IDS) != 25 or len(set(Q38_P1_PUBLIC_GAME_IDS)) != 25:
        raise RuntimeError("Q38 P1 public game list must contain exactly 25 unique games.")
    if not bm.games:
        raise RuntimeError("benchmark_initial.pkl contains no template public game.")

    import taaf.game_api

    template_game = bm.games[0]
    arcade_spec = getattr(template_game, "arcade_spec", None)
    if arcade_spec is None:
        arcade_spec = getattr(template_game, "_arcade_spec", None)
    if arcade_spec is None:
        raise RuntimeError(
            "Could not recover the public ArcadeSpec from benchmark_initial.pkl; "
            "cannot construct the 25-game Q38 P1 evaluation set."
        )

    bm.games = [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=arcade_spec)
        for game_id in Q38_P1_PUBLIC_GAME_IDS
    ]
    bm.n_passes = 1
    bm.game_weights = None

    # These already match Q38 P1 in the source notebook; set them explicitly so the
    # intended evaluation configuration is visible and stable.
    if hasattr(bm.solver, "concurrency"):
        bm.solver.concurrency = 28
    if hasattr(bm.solver, "max_runtime_s_per_game"):
        bm.solver.max_runtime_s_per_game = 7920.0

    bm.label = f"{bm.label}-25g-p1"
    print(f"Public evaluation override: {len(bm.games)} games × {bm.n_passes} pass = {len(bm.games) * bm.n_passes} runs")
    print("Public evaluation concurrency:", getattr(bm.solver, "concurrency", None))
    print("Public per-game runtime cap (s):", getattr(bm.solver, "max_runtime_s_per_game", None))


In [ ]:
# =====================================================================
# BATTLE build: stock Duck plus ONE addition -- the turn-budget hint.
#
# Measured on our 30-game testbed, three runs each, read through
# scripts/rhae.py (never summary.txt, which flushes on a timer and lied
# about two of these runs):
#     stock                RHAE 2.49, deepest 3 levels
#     stock + this hint    RHAE 5.87 / 5.17 / 11.22, deepest 5 -- one game
#                          cleared outright, which had never happened
# Threshold was written before the runs: median >= 4.0. Median came to 5.87.
#
# The hint is applied OUTSIDE any `true_submission` guard on purpose. In the
# lab kernel it sat inside one, next to the own-games swap, and would have
# been skipped by the competition rerun -- shipping plain stock while the log
# said otherwise. Phase A keeps duck's own public 25 for the same reason: a
# submission build must not depend on a path the rerun never takes.
#
# Honest limit: the MECHANISM is unknown. Batching did not rise (8-9% of
# action() calls carry 3+ moves either way), so whatever the hint does, it is
# not what its own wording asks for. Measured on our own games, which flatter
# any agent whose search cracks them; transfer to the hidden set is untested.
# =====================================================================
import inference.agent.tool_agent as _bat_agent
_bat_before = len(_bat_agent.PYTHON_ADDENDUM)
_bat_agent.PYTHON_ADDENDUM = _bat_agent.PYTHON_ADDENDUM + "\n- HARNESS AIDS (read them, never recompute them). Each turn's prompt carries: [STATE] -- level, step, board size and the object table (letter, size, rows, cols) of the current board; [DIFF] -- exactly what changed since your previous call (cells, colour transitions, components that appeared, vanished or moved), 'zero' means nothing changed: out of reach, not interactive, or still animating -- keep your hypothesis; [HUD] -- a bar that shrinks per action and the remaining count, every plan must finish inside it; [NOTICE] -- zero-diff streak; [SHORT] -- the previous call spent fewer than 5 actions.\\n- HELPERS preloaded in every python call (pure functions over `current_frame`, letters as in [STATE]): grid() -> list of row strings; cells(letter) -> [(row, col)]; objects(min_size=1, max_items=60) -> [{letter, size, r0, c0, r1, c1}] largest first; find_path(start, goal, passable, step=1) -> list of UP/DOWN/LEFT/RIGHT moves or None (BFS over cells; passable = set of letters or a function of a letter; goal = (row, col) or a list of them; step = cells per key press); frame_diff() -> {changed, bbox, moved, appeared, vanished} between previous_frame and current_frame; probe(action) -> executes ONE action and returns its frame_diff (use it once per unknown control to learn its reach); move_until(action, max_steps=30) -> repeats one action until the board stops changing or the level or game ends. Never count cells in your head: locate with objects()/cells(), plan with find_path(), then action(path).\\n- EVENT PROTOCOL (harness-enforced). On the first turn of a level, after a zero-diff call and after a game over, the first python call that executes actions MUST begin with:\\n  PROTOCOL = {\\n    'win': '...',    # what exactly ends this level, derived before moving: every X inside a Y of matching shape or colour; all cups filled; one piece left; N of M\\n    'rule': '...',   # the mechanic you believe, incl. the reach of each control (cells per press, what a click on an object does, how far a shove carries -- possibly through walls), what kills, what moves by itself and its period, whether a failed try costs one of N attempts\\n    'plan': '...',   # the sequence this call executes and why\\n  }\\n  Inspection-only calls never need it. On routine turns just act.\\n- ACT IN BATCHES: a call that executes actions should run 5 or more of them (a find_path() result, or a loop that calls action(...) step by step and checks frame_diff() between steps); one or two actions per call is only right for a single probe() of an unknown control, or right after a level ended.\\n- EXPENSIVE PROBES: if a failed try costs an attempt or a life (a counter that drops, a reset after a wrong move), simulate the outcome in code from the rule you hold before acting; never test by trying.\\n\n"
print(f"battle: hint 'lean' applied -- addendum {_bat_before} -> "
      f"{len(_bat_agent.PYTHON_ADDENDUM)} chars (applies in Phase A AND rerun)")
import os as _bat_os; _bat_os.environ['ATLAS_LEAN_AUTOPROBE'] = '0'
import os as _bat_os; _bat_os.environ['ATLAS_LEAN_THINK_BUDGET'] = '1024'
exec('\nimport re as _ln_re, json as _ln_json\nfrom collections import deque as _ln_deque, Counter as _ln_Counter\nimport inference.agent.tool_agent as _ln_agent\n\ndef _ln_stable_hash(s):\n    h = 1469598103934665603\n    for ch in s:\n        h = ((h ^ ord(ch)) * 1099511628211) & 0xFFFFFFFFFFFFFFFF\n    return h\n_LN_HEAD = _ln_re.compile(r"^\\s*(?:#[^\\n]*\\n|\\s*\\n)*PROTOCOL\\s*=\\s*\\{", _ln_re.S)\n_LN_ACTS = _ln_re.compile(r"\\baction\\s*\\(")\n_LN_MIN_BATCH = 5\n_LN_DROP = ("Only tool: `python`.", "Only letter-coded board views", "Keep tool output compact",\n            "For the most recent change, compare", "Use Python to inspect the evidence",\n            "Maintain a compact working world model", "Below you are provided with the current world model",\n            "You may call `action(actions)` more than once", "Ground yourself in `current_frame`",\n            "Focus on what changed most recently", "When ready, call `action(actions)`",\n            "If you include assistant text before a tool call", "When calling `python`, emit exactly")\n\n_LN_HELPERS = r\'\'\'\nfrom collections import deque as _h_deque\n_H_MOVES = {\'UP\': (-1, 0), \'DOWN\': (1, 0), \'LEFT\': (0, -1), \'RIGHT\': (0, 1)}\ndef _h_stable_hash(s):\n    h = 1469598103934665603\n    for ch in s:\n        h = ((h ^ ord(ch)) * 1099511628211) & 0xFFFFFFFFFFFFFFFF\n    return h\n_h_orig_action = action\ndef action(acts, force=False):\n    res = _h_orig_action(acts)\n    try:\n        _flag = _H_LOOP_BREAK          # defined by the harness only when ATLAS_LEAN_LOOP_BREAK=1 (globals() is not a sandbox builtin)\n        seen = _H_SEEN\n        if \'loop\' not in res and _flag and current_frame is not None:\n            hs = _h_stable_hash(str(current_frame.ascii))\n            if hs in seen and not force:\n                res[\'loop\'] = True\n                res[\'note\'] = \'the board returned to a state seen earlier in this game; this sequence undoes itself -- stop and change the plan (pass force=True to override)\'\n            seen.add(hs)\n    except Exception:\n        pass\n    return res\ndef grid(frame=None):\n    f = current_frame if frame is None else frame\n    return [] if f is None else str(f.ascii).split(\'\\n\')\ndef cells(letter, frame=None):\n    g = grid(frame)\n    return [(r, c) for r in range(len(g)) for c in range(len(g[r])) if g[r][c] == letter]\ndef _h_components(g):\n    rows = len(g); seen = set(); out = []\n    for r0 in range(rows):\n        for c0 in range(len(g[r0])):\n            if (r0, c0) in seen:\n                continue\n            ch = g[r0][c0]; q = _h_deque([(r0, c0)]); seen.add((r0, c0)); n = 0\n            rmin = rmax = r0; cmin = cmax = c0\n            while q:\n                r, c = q.popleft(); n += 1\n                if r < rmin: rmin = r\n                if r > rmax: rmax = r\n                if c < cmin: cmin = c\n                if c > cmax: cmax = c\n                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):\n                    nr, nc = r + dr, c + dc\n                    if 0 <= nr < rows and 0 <= nc < len(g[nr]) and (nr, nc) not in seen and g[nr][nc] == ch:\n                        seen.add((nr, nc)); q.append((nr, nc))\n            out.append({\'letter\': ch, \'size\': n, \'r0\': rmin, \'c0\': cmin, \'r1\': rmax, \'c1\': cmax})\n    return out\ndef objects(frame=None, min_size=1, max_items=60):\n    comps = [o for o in _h_components(grid(frame)) if o[\'size\'] >= min_size]\n    comps.sort(key=lambda o: -o[\'size\'])\n    return comps[:max_items]\ndef frame_diff(before=None, after=None):\n    a = grid(previous_frame if before is None else before); b = grid(current_frame if after is None else after)\n    if not a or not b:\n        return {\'changed\': None, \'note\': \'no previous frame\'}\n    changed = [(r, c) for r in range(min(len(a), len(b))) for c in range(min(len(a[r]), len(b[r]))) if a[r][c] != b[r][c]]\n    key = lambda o: (o[\'letter\'], o[\'size\'], o[\'r1\'] - o[\'r0\'], o[\'c1\'] - o[\'c0\'])\n    ca = {}\n    for o in _h_components(a):\n        ca.setdefault(key(o), []).append((o[\'r0\'], o[\'c0\']))\n    moved = []; appeared = []\n    for o in _h_components(b):\n        k = key(o); pos = (o[\'r0\'], o[\'c0\'])\n        if k in ca and ca[k]:\n            if pos in ca[k]:\n                ca[k].remove(pos)\n            else:\n                src = ca[k].pop(0)\n                moved.append({\'letter\': o[\'letter\'], \'size\': o[\'size\'], \'from\': src, \'to\': pos, \'delta\': (pos[0] - src[0], pos[1] - src[1])})\n        else:\n            appeared.append({\'letter\': o[\'letter\'], \'size\': o[\'size\'], \'at\': pos})\n    vanished = [{\'letter\': k[0], \'size\': k[1], \'at\': p} for k, ps in ca.items() for p in ps]\n    bbox = None\n    if changed:\n        bbox = (min(r for r, _ in changed), min(c for _, c in changed), max(r for r, _ in changed), max(c for _, c in changed))\n    return {\'changed\': len(changed), \'bbox\': bbox, \'moved\': moved[:8], \'appeared\': appeared[:8], \'vanished\': vanished[:8]}\ndef probe(act):\n    res = action([act] if isinstance(act, (str, dict)) else act)\n    d = frame_diff()\n    d[\'result\'] = {k: res.get(k) for k in (\'board_changed\', \'level_completed\', \'game_over\', \'executed_count\', \'stop_reason\') if k in res}\n    return d\ndef move_until(act, max_steps=30):\n    steps = 0; last = None\n    for _ in range(max_steps):\n        res = action([act]); steps += 1; last = frame_diff()\n        if res.get(\'game_over\') or res.get(\'level_completed\') or res.get(\'done\') or not res.get(\'board_changed\'):\n            break\n    return {\'steps\': steps, \'last_diff\': last}\ndef _h_background(g, frac=0.15):\n    from collections import Counter as _h_Counter\n    n = sum(len(r) for r in g) or 1\n    return {ch for ch, k in _h_Counter(ch for r in g for ch in r).items() if k >= frac * n}\ndef move_to(target, me=None, passable=None, step=1, max_len=200):\n    g = grid()\n    if me is None:\n        d = frame_diff()\n        mv = d.get(\'moved\') if isinstance(d, dict) else None\n        if not mv:\n            return {\'error\': \'move_to: pass me=(row, col) or me=letter -- no object moved in the last diff\'}\n        me = mv[0][\'to\']; me_letter = mv[0][\'letter\']\n    if isinstance(me, str):\n        cs = cells(me)\n        if not cs:\n            return {\'error\': \'move_to: no cell with letter \' + me}\n        me_letter = me; me = cs[0]\n    else:\n        me = tuple(me); me_letter = g[me[0]][me[1]]\n    if isinstance(target, str):\n        goals = cells(target); tl = target\n    elif isinstance(target, (list, set, frozenset)) and target and isinstance(next(iter(target)), (tuple, list)):\n        goals = [tuple(x) for x in target]; tl = None\n    else:\n        goals = [tuple(target)]; tl = None\n    if not goals:\n        return {\'error\': \'move_to: target not found on the board\'}\n    if tl is None:\n        tl = g[goals[0][0]][goals[0][1]]\n    ok = set(passable) if passable else _h_background(g)\n    ok = ok | {me_letter, tl}\n    path = find_path(me, goals, ok, step=step)\n    if path is None:\n        return {\'error\': \'move_to: no path\', \'passable_used\': sorted(ok), \'from\': me, \'goals\': goals[:5]}\n    path = path[:max_len]\n    res = action(path)\n    return {\'path\': path, \'executed\': res.get(\'executed_count\', len(path)), \'result\': {k: res.get(k) for k in (\'board_changed\', \'level_completed\', \'game_over\', \'stop_reason\') if k in res}, \'diff\': frame_diff()}\ndef find_path(start, goal, passable, step=1, moves=None, frame=None):\n    g = grid(frame); rows = len(g)\n    if callable(passable):\n        ok = passable\n    else:\n        allowed = set(passable); ok = lambda ch: ch in allowed\n    if isinstance(goal, (list, set, frozenset)) and goal and isinstance(next(iter(goal)), (tuple, list)):\n        goals = set(tuple(x) for x in goal)\n    else:\n        goals = {tuple(goal)}\n    mv = moves or _H_MOVES\n    start = tuple(start); prev = {start: None}; q = _h_deque([start])\n    while q:\n        cur = q.popleft()\n        if cur in goals:\n            path = []\n            while prev[cur] is not None:\n                parent, m = prev[cur]; path.append(m); cur = parent\n            return path[::-1]\n        for name, (dr, dc) in mv.items():\n            nr, nc = cur[0] + dr * step, cur[1] + dc * step\n            if not (0 <= nr < rows and 0 <= nc < len(g[nr])):\n                continue\n            if any(not ok(g[cur[0] + dr * k][cur[1] + dc * k]) for k in range(1, step + 1)):\n                continue\n            if (nr, nc) not in prev:\n                prev[(nr, nc)] = (cur, name); q.append((nr, nc))\n    return None\n\'\'\'\n\ndef _ln_rows(frame):\n    if frame is None:\n        return None\n    a = getattr(frame, "ascii", None)\n    if isinstance(a, str) and a:\n        return a.split("\\n")\n    g = getattr(frame, "grid", None)\n    if g is None:\n        return None\n    return ["".join("ABCDEFGHIJKLMNOP"[int(v) % 16] for v in r) for r in g]\n\ndef _ln_components(grid):\n    rows = len(grid); seen = set(); comps = []\n    for r0 in range(rows):\n        for c0 in range(len(grid[r0])):\n            if (r0, c0) in seen:\n                continue\n            col = grid[r0][c0]; seen.add((r0, c0))\n            q = _ln_deque([(r0, c0)]); cells = 0; rmin = rmax = r0; cmin = cmax = c0\n            while q:\n                r, c = q.popleft(); cells += 1\n                rmin = min(rmin, r); rmax = max(rmax, r); cmin = min(cmin, c); cmax = max(cmax, c)\n                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):\n                    nr, nc = r + dr, c + dc\n                    if 0 <= nr < rows and 0 <= nc < len(grid[nr]) and (nr, nc) not in seen and grid[nr][nc] == col:\n                        seen.add((nr, nc)); q.append((nr, nc))\n            comps.append((col, cells, rmin, cmin, rmax, cmax))\n    return comps\n\ndef _ln_state_text(rows, level, step, max_items=24):\n    if not rows:\n        return "[STATE] no frame."\n    comps = sorted(_ln_components(rows), key=lambda t: -t[1])\n    counts = _ln_Counter(ch for r in rows for ch in r)\n    head = f"[STATE] level {level}, step {step}, board {len(rows)}x{max(len(r) for r in rows)}; letters: " + \\\n           ", ".join(f"{k} {v}" for k, v in counts.most_common())\n    items = [f"{col} {cells} r{rmin}-{rmax} c{cmin}-{cmax}" for col, cells, rmin, cmin, rmax, cmax in comps[:max_items]]\n    more = f" (+{len(comps) - max_items} smaller)" if len(comps) > max_items else ""\n    return head + f"\\n  objects (letter size rows cols), largest first, {len(comps)} total{more}: " + "; ".join(items) + "."\n\ndef _ln_bar(prev, cur):\n    out = []\n    def runs(line):\n        res = []; i = 0\n        while i < len(line):\n            j = i\n            while j < len(line) and line[j] == line[i]:\n                j += 1\n            res.append((line[i], i, j - i)); i = j\n        return res\n    def check(name, idx, a, b):\n        if a == b or len(a) != len(b):\n            return\n        ra, rb = runs(a), runs(b)\n        if len(ra) != len(rb) or len(ra) > 4:\n            return\n        diffs = [(x, y) for x, y in zip(ra, rb) if x != y]\n        if len(diffs) != 2:\n            return\n        (ca, sa, la), (cb, sb, lb) = diffs[0]\n        (ca2, sa2, la2), (cb2, sb2, lb2) = diffs[1]\n        if ca == cb and ca2 == cb2 and la > lb and la2 < lb2 and la + la2 == lb + lb2:\n            out.append((name, idx, ca, la, lb))\n    for r in range(min(len(prev), len(cur))):\n        check("row", r, prev[r], cur[r])\n    cols = min(max((len(r) for r in prev), default=0), max((len(r) for r in cur), default=0))\n    for c in range(cols):\n        a = tuple(row[c] for row in prev if c < len(row)); b = tuple(row[c] for row in cur if c < len(row))\n        check("col", c, a, b)\n    return out\n\ndef _ln_diff_text(prev, cur, executed):\n    if prev is None or cur is None:\n        return "[DIFF] no previous frame yet."\n    changed = []\n    for r in range(min(len(prev), len(cur))):\n        pr, cr = prev[r], cur[r]\n        for c in range(min(len(pr), len(cr))):\n            if pr[c] != cr[c]:\n                changed.append((r, c, pr[c], cr[c]))\n    if not changed:\n        return "[DIFF] zero -- no cell changed since your previous call."\n    rs = [x[0] for x in changed]; cs = [x[1] for x in changed]\n    trans = _ln_Counter((a, b) for _, _, a, b in changed).most_common(4)\n    lines = [f"[DIFF] {len(changed)} cells changed in rows {min(rs)}-{max(rs)}, cols {min(cs)}-{max(cs)}; "\n             f"colour transitions: " + ", ".join(f"{a}->{b} x{n}" for (a, b), n in trans) + "."]\n    try:\n        key = lambda t: (t[0], t[1], t[4] - t[2] + 1, t[5] - t[3] + 1)\n        pc = _ln_Counter(key(t) for t in _ln_components(prev) if t[1] <= 400)\n        cc = _ln_Counter(key(t) for t in _ln_components(cur) if t[1] <= 400)\n        gone = list((pc - cc).elements()); new = list((cc - pc).elements())\n        if gone or new:\n            fmt = lambda k: f"{k[0]} {k[2]}x{k[3]} ({k[1]} cells)"\n            parts = []\n            if gone:\n                parts.append("vanished: " + ", ".join(fmt(k) for k in gone[:5]) + (" ..." if len(gone) > 5 else ""))\n            if new:\n                parts.append("appeared: " + ", ".join(fmt(k) for k in new[:5]) + (" ..." if len(new) > 5 else ""))\n            lines.append("  components " + "; ".join(parts) + ".")\n        moved = []\n        if gone and new:\n            pmap = {}\n            for t in _ln_components(prev):\n                pmap.setdefault(key(t), []).append((t[2], t[3]))\n            for t in _ln_components(cur):\n                k = key(t)\n                if k in pmap and pmap[k] and (t[2], t[3]) not in pmap[k]:\n                    r0, c0 = pmap[k].pop(0)\n                    moved.append(f"{t[0]} {k[2]}x{k[3]} from (r{r0},c{c0}) to (r{t[2]},c{t[3]})")\n        if moved:\n            lines.append("  moved: " + "; ".join(moved[:4]) + ".")\n    except Exception:\n        pass\n    try:\n        for name, idx, letter, la, lb in _ln_bar(prev, cur)[:2]:\n            rate = (la - lb) / max(1, executed or 1)\n            left = int(lb / rate) if rate > 0 else lb\n            lines.append(f"[HUD] {name} {idx}: a {letter} bar shrank {la}->{lb} cells "\n                         f"({la - lb} for {executed or \'?\'} actions) -> about {left} actions left if it is a budget.")\n    except Exception:\n        pass\n    return "\\n".join(lines)\n\n_ln_orig_prompt = _ln_agent.ToolAgent._build_user_prompt\ndef _ln_prompt(self, action_num, *args, **kwargs):\n    text = _ln_orig_prompt(self, action_num, *args, **kwargs)\n    kept = [l for l in text.split("\\n") if not any(l.startswith(pfx) for pfx in _LN_DROP)]\n    text = "\\n".join(kept)\n    frame = kwargs.get("current_frame"); summary = kwargs.get("previous_step_summary") or {}\n    rows = _ln_rows(frame); level = getattr(frame, "level", None); step = getattr(frame, "step", None)\n    try:\n        executed = int(summary.get("executed_count") or 0)\n    except (TypeError, ValueError):\n        executed = 0\n    prev = getattr(self, "_ln_prev_rows", None); prev_level = getattr(self, "_ln_prev_level", None)\n    new_level = prev_level is None or (level is not None and level != prev_level) or bool(summary.get("level_transition"))\n    zero = executed > 0 and not summary.get("board_changed") and not summary.get("level_transition")\n    streak = getattr(self, "_ln_zero_streak", 0)\n    if zero:\n        streak += 1\n    elif executed > 0:\n        streak = 0\n    self._ln_zero_streak = streak\n    extra = [_ln_state_text(rows, level, step)]\n    if new_level:\n        extra.append("[DIFF] new level -- rebuild win and rule from scratch; the previous level\'s rule is only a hypothesis here.")\n    else:\n        extra.append(_ln_diff_text(prev, rows, executed))\n    if streak >= 1:\n        extra.append(f"[NOTICE] {streak} zero-diff call{\'s\' if streak > 1 else \'\'} in a row: out of reach, not interactive, or still animating. Keep the hypothesis; look again before changing it.")\n    ctrl = getattr(self, "_ln_controls", None)\n    if ctrl and ctrl[0] == level:\n        extra.append(ctrl[1])\n    probed = getattr(self, "_ln_probe_just_ran", False); self._ln_probe_just_ran = False\n    if executed and executed < _LN_MIN_BATCH and not summary.get("level_transition") and not summary.get("game_over") and not new_level and not probed:\n        extra.append(f"[SHORT] the previous call spent only {executed} action{\'s\' if executed > 1 else \'\'}; batch 5+ (find_path + action) or loop with checks unless this was a single probe.")\n    event = new_level or zero or bool(summary.get("game_over"))\n    self._ln_protocol_required = event\n    self._ln_event_turn = event\n    try:\n        hud_left = None\n        for name, idx, letter, la, lb in (_ln_bar(prev, rows) if (prev is not None and rows is not None) else [])[:1]:\n            rate = (la - lb) / max(1, executed or 1)\n            hud_left = int(lb / rate) if rate > 0 else None\n        self._ln_hud_left = hud_left\n    except Exception:\n        self._ln_hud_left = None\n    try:\n        if rows is not None:\n            hsh = _ln_stable_hash("\\n".join(rows))\n            seen = getattr(self, "_ln_seen", None)\n            if seen is None or new_level:\n                seen = set(); self._ln_seen = seen; self._ln_seen_turn = {}\n            if hsh in seen and not new_level and executed > 0:\n                extra.append(f"[LOOP] the board is identical to a state you already saw {action_num - self._ln_seen_turn.get(hsh, action_num)} calls ago -- the last actions undid earlier ones; do not repeat that cycle.")\n            seen.add(hsh); self._ln_seen_turn.setdefault(hsh, action_num)\n    except Exception:\n        pass\n    if event:\n        extra.append("[THINK] Event turn: the first python call that executes actions must start with PROTOCOL = {\'win\': ..., \'rule\': ..., \'plan\': ...}; think it through.")\n    self._ln_prev_rows = rows; self._ln_prev_level = level\n    return text + "\\n" + "\\n".join(extra)\n_ln_agent.ToolAgent._build_user_prompt = _ln_prompt\n\n_LN_PROBE_CODE = r\'\'\'\n_ctrl = []\nfor _a in __ARROWS__:\n    _r = action([_a])\n    _d = frame_diff()\n    _mv = _d.get(\'moved\') or []\n    if _r.get(\'game_over\') or _r.get(\'level_completed\') or _r.get(\'done\'):\n        _ctrl.append(_a + \': \' + (\'GAME OVER\' if _r.get(\'game_over\') else \'level changed\')); break\n    if not _r.get(\'board_changed\') and not _d.get(\'changed\'):\n        _ctrl.append(_a + \': zero diff\')\n    elif _mv:\n        _ctrl.append(_a + \': \' + \'; \'.join(\'%s %d cells moved (%d,%d)\' % (m[\'letter\'], m[\'size\'], m[\'delta\'][0], m[\'delta\'][1]) for m in _mv[:3]))\n    else:\n        _ctrl.append(_a + \': %d cells changed%s\' % (_d.get(\'changed\') or 0, (\', appeared \' + \',\'.join(x[\'letter\'] for x in (_d.get(\'appeared\') or [])[:3])) if _d.get(\'appeared\') else \'\'))\nprint(\'[CONTROLS] harness pressed each arrow once (these actions are spent): \' + \' | \'.join(_ctrl))\n\'\'\'\n_ln_orig_analyze = _ln_agent.ToolAgent.analyze\ndef _ln_analyze(self, state_path, action_num, valid_actions=None, step_env=None, *args, **kwargs):\n    import os as _ln_os2\n    level = None\n    try:\n        if _ln_os2.environ.get("ATLAS_LEAN_AUTOPROBE", "1").strip().lower() not in ("0", "false", "no") and state_path.exists():\n            self._ensure_session(state_path)\n            self._step_env_callback = step_env\n            self._current_valid_actions = _ln_agent._normalize_valid_actions(valid_actions)\n            frame, _hist = _ln_agent.load_runtime_state(state_path)\n            level = getattr(frame, "level", None)\n            arrows = [a for a in ("UP", "DOWN", "LEFT", "RIGHT") if a in set(self._current_valid_actions)]\n            if level is not None and getattr(self, "_ln_probe_level", None) != level and arrows and step_env is not None:\n                self._ln_probe_level = level\n                res = self._run_python_tool(state_path, {"code": _LN_PROBE_CODE.replace("__ARROWS__", repr(arrows))})\n                text = getattr(res, "content", "") or ""\n                m = _ln_re.search(r"\\[CONTROLS\\][^\\n]*", text)\n                self._ln_controls = (level, m.group(0) if m else "[CONTROLS] probe ran but produced no summary")\n                self._ln_probe_just_ran = True\n                self._ln_zero_streak = 0\n    except Exception as exc:\n        self._ln_controls = (level, "[CONTROLS] auto-probe failed: %s: %s" % (type(exc).__name__, str(exc)[:120]))\n    return _ln_orig_analyze(self, state_path, action_num, valid_actions, step_env, *args, **kwargs)\n_ln_agent.ToolAgent.analyze = _ln_analyze\n\n_ln_orig_run = _ln_agent.ToolAgent._run_python_tool\ndef _ln_run(self, state_path, arguments):\n    code = str((arguments or {}).get("code", "") or "")\n    if getattr(self, "_ln_protocol_required", False) and _LN_ACTS.search(code):\n        if not _LN_HEAD.match(code):\n            return _ln_agent._ToolDispatchResult(_ln_json.dumps({"error": (\n                "Rejected before execution: this is an event turn (new level, zero diff or game over), so the first "\n                "python call that executes actions must START with PROTOCOL = {\'win\': ..., \'rule\': ..., \'plan\': ...}. "\n                "Inspection-only code needs no header. No action was spent.")}, indent=2))\n        self._ln_protocol_required = False\n    import os as _ln_os3\n    if _LN_ACTS.search(code):\n        lits = _LN_ACT_LIST.findall(code)\n        n_lit = sum(len(_LN_ACT_ITEM.findall(body)) for body in lits)\n        has_loop = bool(_ln_re.search(r"^\\s*(for|while)\\b", code, _ln_re.M)) or "move_to(" in code or "move_until(" in code or "*" in "".join(lits)\n        if _ln_os3.environ.get("ATLAS_LEAN_MIN_BATCH_HARD", "").strip() in ("1", "true", "yes") and lits and not has_loop and 0 < n_lit < _LN_MIN_BATCH and not getattr(self, "_ln_event_turn", False):\n            return _ln_agent._ToolDispatchResult(_ln_json.dumps({"error": (\n                f"Rejected before execution: this call would spend only {n_lit} action(s). Batch 5 or more (a computed path, a loop that "\n                "checks frame_diff() between steps, or move_to), or explain in PROTOCOL why a single probe is needed. No action was spent.")}, indent=2))\n        left = getattr(self, "_ln_hud_left", None)\n        if _ln_os3.environ.get("ATLAS_LEAN_HUD_BLOCK", "").strip() in ("1", "true", "yes") and left is not None and lits and not has_loop and n_lit > left:\n            return _ln_agent._ToolDispatchResult(_ln_json.dumps({"error": (\n                f"Rejected before execution: the plan has {n_lit} actions but the [HUD] bar leaves about {left}. Plan inside the remaining budget. No action was spent.")}, indent=2))\n    if _ln_os3.environ.get("ATLAS_LEAN_LOOP_BREAK", "").strip() in ("1", "true", "yes"):\n        seen = sorted(getattr(self, "_ln_seen", set()))\n        arguments = dict(arguments); arguments["code"] = "_H_SEEN = set(" + repr(seen) + ")\\n_H_LOOP_BREAK = True\\n" + code\n    return _ln_orig_run(self, state_path, arguments)\n_ln_agent.ToolAgent._run_python_tool = _ln_run\n_LN_ACT_LIST = _ln_re.compile(r"action\\s*\\(\\s*\\[(.*?)\\]\\s*\\)", _ln_re.S)\n_LN_ACT_ITEM = _ln_re.compile(r"\'[A-Z0-9_]+\'|\\"[A-Z0-9_]+\\"|\\{[^{}]*\\}")\n\n_ln_orig_sandbox = _ln_agent.run_sandboxed_python\ndef _ln_sandbox(*args, **kwargs):\n    code = kwargs.get("code")\n    if isinstance(code, str):\n        kwargs["code"] = _LN_HELPERS + "\\n" + code\n    return _ln_orig_sandbox(*args, **kwargs)\n_ln_agent.run_sandboxed_python = _ln_sandbox\n\n_ln_orig_payload = _ln_agent.build_chat_payload\ndef _ln_payload(*args, **kwargs):\n    msgs = kwargs.get("messages") or []\n    think = False\n    for m in reversed(msgs):\n        if isinstance(m, dict) and m.get("role") == "user":\n            c = m.get("content")\n            if isinstance(c, list):\n                c = " ".join(str(part.get("text", "")) for part in c if isinstance(part, dict))\n            think = "[THINK]" in str(c or "")\n            break\n    marker = think          # event turn = the prompt carries [THINK]; kept before the default override below\n    import os as _ln_os\n    # DEFAULT = variant B (pod 04.09: RHAE 5.44 vs 0.25 for events-only thinking): think on every turn.\n    # ATLAS_LEAN_THINK_EVENTS_ONLY=1 restores variant A (thinking only on [THINK] turns) for experiments.\n    if _ln_os.environ.get("ATLAS_LEAN_THINK_EVENTS_ONLY", "").strip().lower() not in ("1", "true", "yes"):\n        think = True\n    budget = _ln_os.environ.get("ATLAS_LEAN_THINK_BUDGET", "").strip()\n    if budget.isdigit() and int(budget) > 0:\n        # variant C (Gemini r.18 as written): think on every turn, but routine turns get a token budget;\n        # event turns ([THINK] in the prompt) think without a cap. Needs vLLM started with --reasoning-config.\n        event = marker; think = True\n        payload = _ln_orig_payload(*args, **dict(kwargs, thinking=True))\n        if not event:\n            payload["thinking_token_budget"] = int(budget)\n        return payload\n    effort = _ln_os.environ.get("ATLAS_LEAN_REASONING_EFFORT", "").strip().lower()\n    if effort in ("none", "low", "medium", "high"):\n        event = marker\n        payload = _ln_orig_payload(*args, **dict(kwargs, thinking=True))\n        if not event:\n            payload["reasoning_effort"] = effort\n        return payload\n    kwargs["thinking"] = bool(think)\n    return _ln_orig_payload(*args, **kwargs)\n_ln_agent.build_chat_payload = _ln_payload\nprint("lean: harness wrappers installed (compact prompt + [STATE]/[CONTROLS]/[DIFF]/[HUD]/[NOTICE]/[SHORT], sandbox helpers incl. move_to, auto-probe of arrows on new levels, event PROTOCOL)")\n')

if not true_submission:
    if hasattr(bm.solver, "max_runtime_s_per_game"):
        bm.solver.max_runtime_s_per_game = 7920.0
        print(f"battle: Phase A cap {bm.solver.max_runtime_s_per_game}s on duck's public 25")
    # ---- test variant: restart the stock vLLM with --reasoning-config (Phase A only; enables thinking_token_budget)
    import json as _vj, os as _vo, signal as _vs, subprocess as _vp, sys as _vsys, time as _vt, urllib.request as _vu
    _pidf = WORKING_DIR / "vllm-openai-server.pid"
    try:
        _pid = int(_pidf.read_text().strip()); _vo.kill(_pid, _vs.SIGTERM); print("battle: sent SIGTERM to stock vLLM pid", _pid)
    except Exception as _e:
        print("battle: could not kill stock vLLM:", _e)
    for _ in range(90):
        try:
            _vu.urlopen("http://127.0.0.1:1234/v1/models", timeout=3); _vt.sleep(2)
        except Exception:
            break
    _senv = _load_setup_env()
    _model = _senv.get("TAAF_QWEN_MODEL_PATH") or _vo.environ.get("TAAF_QWEN_MODEL_PATH")
    _served = _senv.get("TAAF_QWEN_SERVED_MODEL_NAME") or _vo.environ.get("TAAF_QWEN_SERVED_MODEL_NAME") or "Qwen/Qwen3.8-27B-FP8"
    _cmd = [_vsys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", str(_model), "--served-model-name", _served,
            "--host", "127.0.0.1", "--port", "1234", "--tensor-parallel-size", "1", "--enable-auto-tool-choice",
            "--tool-call-parser", "qwen3_coder", "--generation-config", "vllm", "--enable-prefix-caching",
            "--default-chat-template-kwargs", '{"preserve_thinking": true}', "--reasoning-parser", "qwen3",
            "--max-model-len", "65536", "--reasoning-config", '{"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"}']
    _vlog = open(WORKING_DIR / "vllm-openai-server-2.log", "w", encoding="utf-8")
    _proc = _vp.Popen(_cmd, env=_command_env(), stdout=_vlog, stderr=_vp.STDOUT)
    _pidf.write_text(str(_proc.pid))
    _ok = False
    for _ in range(180):
        if _proc.poll() is not None:
            break
        try:
            _vu.urlopen("http://127.0.0.1:1234/v1/models", timeout=5); _ok = True; break
        except Exception:
            _vt.sleep(5)
    print("battle: vLLM restarted with --reasoning-config:", _ok)
    if not _ok:
        raise RuntimeError("vLLM restart with --reasoning-config failed; see vllm-openai-server-2.log")


In [ ]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None

    try:
        await bm.run(
            soft_end_time=soft_end,
            runtime_environment=target,
            minimal_diagnostics=run_as_submission,
        )
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")